# Bangla-LLM: End-to-End Training & Evaluation Notebook

This Jupyter Notebook contains the **complete pipeline** for fine-tuning, evaluating, and exporting the **BanglaSupport-LLM 7B** model on a GPU:

1. **Environment Setup**: Verifies CUDA availability and installs Unsloth QLoRA dependencies.
2. **Dataset Acquisition & Preprocessing**: Downloads `Bangla-Instruct` & `Aya Dataset`, performs NFC Unicode normalization, MinHash LSH deduplication, and structures intent data.
3. **QLoRA Fine-Tuning Execution**: Performs 4-bit NF4 QLoRA fine-tuning using Unsloth & `SFTTrainer` on Qwen2.5 / Qwen3 base models.
4. **Multi-Metric Evaluation**: Generates responses using the fine-tuned model and evaluates performance across **BLEU-4**, **ROUGE-L**, and **BERTScore**.
5. **Dual Weight Export**: Exports merged **Safetensors** for GPU deployment and 4-bit **GGUF** for fast CPU inference.

## Step 1: Install Dependencies

In [ ]:
!pip install --upgrade pip
!pip install --upgrade --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets transformers trl peft bitsandbytes sentencepiece protobuf
!pip install rouge-score nltk bert-score datasketch unicodedata2

In [ ]:
import torch

print("==========================================================")
print("PyTorch Environment Verification")
print("==========================================================")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Device: {device_name}")
    print(f"VRAM Capacity: {vram_gb:.2f} GB")
else:
    print("Warning: Running in CPU mode")

## Step 2: Dataset Pipeline 

In [ ]:
import os
import re
import json
import unicodedata
from datasets import load_dataset, Dataset
from datasketch import MinHash, MinHashLSH
from sklearn.model_selection import train_test_split

INTENT_PATTERNS = {
    "order_status": [r"অর্ডার", r"ট্র্যাক", r"স্ট্যাটাস", r"কোথায় আছে", r"ডেলিভারি কবে", r"স্ট্যাটাস জানতে"],
    "return_policy": [r"রিটার্ন", r"পরিবর্তন", r"ফেরত", r"বদলাব", r"ফেরত দেব"],
    "shipping_fee": [r"ডেলিভারি চার্জ", r"শিপিং", r"ভাড়া", r"চার্জ কত", r"ডেলিভারি ফ্রি"],
    "payment_methods": [r"পেমেন্ট", r"ক্যাশ অন", r"বিকাশ", r"রকেট", r"নগদ", r"কার্ড"],
    "cancellation": [r"বাতিল", r"কেনসেল", r"অর্ডার ক্যানসেল"],
    "refund_status": [r"রিফান্ড", r"টাকা ফেরত", r"পয়সা ফেরত"],
    "account_access": [r"অ্যাকাউন্ট", r"লগইন", r"পাসওয়ার্ড", r"সাইন ইন"],
    "product_inquiry": [r"স্টক", r"সাইজ", r"কালার", r"দাম কত", r"পণ্য", r"কোয়ালিটি"],
    "store_location": [r"ঠিকানা", r"শোরুম", r"লোকেশন", r"দোকান", r"শাখা"],
    "discounts_offers": [r"ডিসকাউন্ট", r"অফার", r"কুপন", r"ছাড়", r"প্রমো"],
    "agent_support": [r"প্রতিনিধি", r"কথা বলতে চাই", r"লাইভ চ্যাট", r"হেল্পলাইন", r"যোগাযোগ"],
    "general_faq": [r"হেল্প", r"সাহায্য", r"তথ্য", r"কীভাবে", r"কেন", r"কি"]
}

def classify_intent(text: str) -> str:
    for intent, patterns in INTENT_PATTERNS.items():
        if any(re.search(pat, text, re.IGNORECASE) for pat in patterns):
            return intent
    return "general_faq"

def normalize_bangla_text(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    return re.sub(r"\s+", " ", text).strip()

print("1. Fetching instruction pairs from dataset...")
raw_samples = []

try:
    ds1 = load_dataset("md-nishat-008/Bangla-Instruct", split="train", streaming=True)
    for i, item in enumerate(ds1):
        if i >= 6000: break
        inst = normalize_bangla_text(item.get("instruction", ""))
        resp = normalize_bangla_text(item.get("response", ""))
        if inst and resp and len(inst) > 5:
            raw_samples.append({
                "instruction": inst,
                "context": "",
                "output": resp,
                "intent": classify_intent(inst)
            })
    print(f"   - Bangla-Instruct: {len(raw_samples)} pairs loaded")
except Exception as e:
    print(f"   - Bangla-Instruct notice: {e}")

try:
    ds2 = load_dataset("CohereForAI/aya_dataset", split="train", streaming=True)
    count = 0
    for item in ds2:
        if item.get("language_code") == "ben" or item.get("language") == "bengali":
            inst = normalize_bangla_text(item.get("inputs", ""))
            resp = normalize_bangla_text(item.get("targets", ""))
            if inst and resp and len(inst) > 5:
                raw_samples.append({
                    "instruction": inst,
                    "context": "",
                    "output": resp,
                    "intent": classify_intent(inst)
                })
                count += 1
                if count >= 6000: break
    print(f"   - Aya Dataset (Bengali): {count} pairs loaded")
except Exception as e:
    print(f"   - Aya Dataset notice: {e}")

print(f"2. Performing MinHash LSH deduplication on {len(raw_samples)} collected pairs...")
lsh = MinHashLSH(threshold=0.85, num_perm=128)
clean_data = []
for idx, sample in enumerate(raw_samples):
    text = sample["instruction"] + " " + sample["output"]
    m = MinHash(num_perm=128)
    for word in text.split():
        m.update(word.encode("utf-8"))
    result = lsh.query(m)
    if not result:
        lsh.insert(f"doc_{idx}", m)
        clean_data.append(sample)

print(f"   - Unique deduplicated pairs: {len(clean_data)}")

output_dir = "../Datasets"
os.makedirs(output_dir, exist_ok=True)
train_samples, val_samples = train_test_split(clean_data, test_size=0.1, random_state=42)

with open(os.path.join(output_dir, "train.jsonl"), "w", encoding="utf-8") as f:
    for item in train_samples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(os.path.join(output_dir, "val.jsonl"), "w", encoding="utf-8") as f:
    for item in val_samples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"3. Saved processed JSONL files in {output_dir}: {len(train_samples)} train / {len(val_samples)} val")

raw_dataset = load_dataset("json", data_files={
    "train": os.path.join(output_dir, "train.jsonl"),
    "validation": os.path.join(output_dir, "val.jsonl")
})
print("✓ Dataset Pipeline Completed Successfully!")

## Step 3: QLoRA Setup & Execution

In [ ]:
import os
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

max_seq_length = 2048
dtype = None 
load_in_4bit = True 

print("Loading 4-bit Base Model with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✓ QLoRA Model Initialized Successfully!")

In [ ]:
import os
import torch
from datasets import load_dataset, Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

output_dir = "checkpoints"
os.makedirs(output_dir, exist_ok=True)

def format_prompts(examples):
    texts = []
    for inst, out in zip(examples["instruction"], examples["output"]):
        prompt = f"<|im_start|>system\nতুমি একজন সহায়ক বাংলা ই-কমার্স গ্রাহক সেবা সহকারী।<|im_end|>\n<|im_start|>user\n{inst}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
        texts.append(prompt)
    return { "text" : texts }

dataset_file = None
for candidate in [
    "../Datasets/train.jsonl",
    "Datasets/train.jsonl"
]:
    if os.path.exists(candidate):
        dataset_file = candidate
        break

if "raw_dataset" in locals():
    train_data_to_format = raw_dataset["train"]
elif dataset_file:
    train_data_to_format = load_dataset("json", data_files={"train": dataset_file})["train"]
else:
    raise RuntimeError("Please run Step 2 (Dataset Pipeline cell) first to generate the dataset!")

formatted_dataset = train_data_to_format.map(format_prompts, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
        bf16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = output_dir,
    ),
)
print(f"✓ SFTTrainer Configured & Ready! Total training samples: {len(formatted_dataset)}")
print("-----------------------------")
print("Starting QLoRA Fine-Tuning...", flush=True)
trainer_stats = trainer.train()
print("✓ Fine-Tuning Completed Successfully!", flush=True)


## Step 4: Multi-Metric Evaluation

In [ ]:
import os
import torch
import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm
from datasets import load_dataset
from unsloth import FastLanguageModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

FastLanguageModel.for_inference(model)
if hasattr(model, "generation_config") and model.generation_config:
    model.generation_config.max_length = None
    model.generation_config.max_new_tokens = 256

val_file = None
for candidate in [
    "../Datasets/val.jsonl",
    "Datasets/val.jsonl"
]:
    if os.path.exists(candidate):
        val_file = candidate
        break

if not val_file:
    raise RuntimeError("Validation dataset (val.jsonl) not found in Datasets directory!")

val_dataset = load_dataset("json", data_files={"val": val_file})["val"]
eval_samples = list(val_dataset)[:50]

predictions = []
references = [x["output"] for x in eval_samples]

print(f"Generating answers from fine-tuned model for {len(eval_samples)} validation samples...", flush=True)
for idx, sample in enumerate(tqdm(eval_samples, desc="Evaluating")): 
    prompt = f"<|im_start|>system\nতুমি একজন সহায়ক বাংলা ই-কমার্স গ্রাহক সেবা সহকারী।<|im_end|>\n<|im_start|>user\n{sample['instruction']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    outputs = model.generate(**inputs, max_new_tokens=256, max_length=None, use_cache=True)
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    predictions.append(generated_text.strip())

smooth = SmoothingFunction().method1
bleu_scores = [sentence_bleu([ref.split()], pred.split(), smoothing_function=smooth) for pred, ref in zip(predictions, references)]
r_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rouge_scores = [r_scorer.score(ref, pred)["rougeL"].fmeasure for pred, ref in zip(predictions, references)]

print("==========================================================")
print("--- Fine-Tuned Model Evaluation Results ---")
print("==========================================================")
print(f"Average BLEU-4 Score:  {sum(bleu_scores)/max(len(bleu_scores), 1):.4f}")
print(f"Average ROUGE-L Score: {sum(rouge_scores)/max(len(rouge_scores), 1):.4f}")


## Step 5: Save Models

In [ ]:
print("Saving Merged 16-bit Safetensors for GPU serving...")
model.save_pretrained_merged("models/BanglaLLM-7B", tokenizer, save_method = "merged_16bit")
print("✓ Saved to Research/models/BanglaLLM-7B/model.safetensors")

try:
    print("Saving 4-bit GGUF model for CPU serving...")
    model.save_pretrained_gguf("models", tokenizer, quantization_method = "q4_k_m")
    print("✓ Saved to Research/models/banglallm-7b-q4_k_m.gguf")
except Exception as e:
    print(f"GGUF Export note: {e}")